# Phase 03 — Exploratory Data Analysis

*Agriculture Risk Monitoring System — Master research-grade portfolio (Project 5).*

**Goal (scope §6.2):** establish a baseline empirical understanding of climate and yield variation across the AAGIS broadacre regions — *where, when, and how much* — and record the optional-crop decision (sorghum / cotton).

**How to run:** execute top-to-bottom on a fresh kernel. The kernel must be the project `.venv` (Python 3.12); do **not** use a base conda kernel. Every figure added in later steps carries a one-sentence interpretation beneath it (scope §11.9).

**Build order (steps):** s01 setup + verified data inventory (this commit) → s02 regional climate climatology → s03 yield trends & dispersion → s04 climate–yield joint view + historical-event sanity → s05 optional-crop decision → s06 closure.


## 0. Setup & verified data inventory

In [1]:
# Bootstrap: put the repo root on sys.path so `from src...` works from notebooks/.
import sys
from pathlib import Path

_REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))

import pandas as pd

from src.viz.style import apply_style, SEED
from src.processing.input_inventory import (
    build_inventory,
    check_invariants,
    reliable_yield_summary,
)

apply_style()
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
print(f"repo root: {_REPO_ROOT}")
print(f"SEED = {SEED}")


repo root: C:\Users\kotae\Documents\Portfolio\project\Project 5\agriculture-risk-monitoring-system
SEED = 42


### 0.1 Product inventory
The Phase 02 analysis-ready inputs and their shapes. This is the same inventory the `phase03_s01_data_inventory.py` orchestrator writes to `outputs/tables/s01_input_inventory.csv`.

In [2]:
inventory = build_inventory()
inventory[["product", "path", "exists", "n_rows", "expected_rows", "row_ok"]]


,product,path,exists,n_rows,expected_rows,row_ok
0,region_mapping,region_mapping.csv,True,32,32.0,True
1,wheat,abares/wheat.csv,True,1114,1114.0,True
2,barley,abares/barley.csv,True,1114,1114.0,True
3,canola,abares/canola.csv,True,1114,1114.0,True
4,wheat_totals,abares/wheat_region_totals.csv,True,1114,1114.0,True
5,barley_totals,abares/barley_region_totals.csv,True,1114,1114.0,True
6,canola_totals,abares/canola_region_totals.csv,True,1114,1114.0,True
7,silo_annual,silo_region_means/annual_1991_2020.csv,True,5400,5400.0,True
8,silo_monthly,silo_region_means/monthly_climatology_1991_202...,True,2160,2160.0,True
9,silo_cell_map,silo_cell_region_map.parquet,True,28577,28577.0,True


### 0.2 Structural invariants
Region-key alignment across the yield (region name) and climate (aagis_code) products, and full coverage of the 20 broadacre regions. The notebook fails loudly here if any Phase 02 output has drifted.

In [3]:
checks = check_invariants()
failed = [c.name for c in checks if not c.ok]
for c in checks:
    print(f"[{'OK ' if c.ok else 'FAIL'}] {c.name}: {c.detail}")
assert not failed, f"structural invariants failed: {failed}"


[OK ] region_mapping_32: 32 rows, 32 unique codes
[OK ] broadacre_count_20: 20 broadacre regions (want 20)
[OK ] silo_annual_structure: 30 regions x 6 vars x 30 years
[OK ] silo_covers_broadacre: all 20 broadacre regions present
[OK ] silo_uncovered_are_pastoral: uncovered=['511', '711'] zones=['Pastoral'] (want 511,711 both Pastoral)
[OK ] silo_region_names_in_mapping: region_name subset of mapping
[OK ] silo_monthly_structure: 30 regions x 6 vars x 12 months
[OK ] wheat_regions_in_mapping: region names subset of mapping
[OK ] wheat_covers_broadacre: all 20 broadacre regions present
[OK ] barley_regions_in_mapping: region names subset of mapping
[OK ] barley_covers_broadacre: all 20 broadacre regions present
[OK ] canola_regions_in_mapping: region names subset of mapping
[OK ] canola_covers_broadacre: all 20 broadacre regions present


### 0.3 Reliable yield windows
Per-commodity yield coverage and the rows that fall before each crop's reliable window (wheat/barley 1990+, canola 1994+ per the RSE gate, scope v5.1 §3.3).

In [4]:
reliable = reliable_yield_summary()
reliable


,commodity,reliable_start,rows,non_na_yield,min_year,max_year,pre_window_rows,pre_window_non_na_yield
0,wheat,1990,1114,747,1990,2024,0,0
1,barley,1990,1114,738,1990,2024,0,0
2,canola,1994,1114,482,1990,2024,122,28


**Interpretation (s01).** The Phase 02 inputs load cleanly and all structural invariants hold: 32 mapped regions, 20 broadacre regions jointly present in the yield and climate products, and the two SILO-uncovered regions are exactly the pastoral 511 / 711. Canola carries 122 pre-1994 rows (28 with a non-NA yield) that the reliable window excludes — a fact carried into the yield EDA (s03) and the optional-crop decision (s05). The data baseline is verified; substantive EDA follows.

## 1. Regional climate climatology (SILO)

*To be populated in s02.* Temperature and rainfall climatologies, seasonality, and interannual variation / trends by AAGIS broadacre region.

**Open decision (gate ①):** the persisted SILO region means currently cover the 1991–2020 reference period only (`silo_region_means/annual_1991_2020.csv`). Trend and variation narrative over the yield window needs a full-record re-aggregation (1990–2024, or 1961–2024) — the full daily SILO archive is already on disk, so no re-download is required. Window choice is decided at the start of s02.

_Each figure here will carry a one-sentence interpretation beneath it._

## 2. Yield trends & dispersion (ABARES)

*To be populated in s03.* Region-level yield trends and dispersion for wheat / barley / canola, reliable-window filtering, and lower-tail (risk-relevant) distributional properties.

**Open decision (gate ②):** RSE weighting — diagnostic-only vs inverse-RSE regression weights (may be deferred to Phase 06).

_Each figure here will carry a one-sentence interpretation beneath it._

## 3. Climate–yield joint view & historical-event sanity

*To be populated in s04.* Region-year alignment of climate and yield, preliminary correlation views, and a sanity overlay of known events — Millennium Drought (2001–2009), the 2018 drought, and the 2013 / 2017 heat — against the SILO / yield series (Task H; formal historical-event validation remains Phase 09).

_Each figure here will carry a one-sentence interpretation beneath it._

## 4. Optional-crop evidence & decision (sorghum / cotton)

*To be populated in s05.* Evidence-led decision on the optional crops, on the basis of (a) ABARES FDP region-year data availability, (b) homogeneity with the rainfed broadacre framing (cotton is irrigated; sorghum is a summer crop with a different climate window), (c) methodological cost, and (d) marginal portfolio signal. The decision is recorded here and in PROJECT_LOG / methodology at closure.


## 5. EDA synthesis — where, when, how much

*To be populated across s02–s05 and consolidated at s06.* The Phase 03 exit-criterion narrative: a clear statement of where, when, and how much climate and yield variation occurs, supported by the figures above.


## Limitations & caveats (running list)

- **ACORN-SAT station truth** is weak for QLD Eastern Darling Downs (no station) and sparse for NSW Central West, VIC Wimmera, VIC Central North, WA South West Coastal; the WA Wheatbelt (521/522) and NSW Central West (122) also lost stations. Flag in any climate-validation narrative.
- **Reliable yield windows:** wheat/barley 1990+, canola 1994+ (RSE gate).
- **SILO region means** are 1991–2020 until the s02 full-record re-aggregation.
- **Observed-vs-derived discipline:** ABARES observed yields are the target; AGFD (Phase 09) is validation only.
